# 4.11 · 决策树回归 / Decision Tree Regression

> **课程定位 / Where this fits**
> 第 11 课，**Part 4 · 监督学习：回归**。
> Lesson 11, **Part 4 · Supervised Regression**.
>
> 5.6 讲过分类树。回归树思路相同——用一连串"是/否"问题把空间切成方块，但**每个叶子预测一个常数（该区域目标的均值）**，切分准则从"不纯度"换成"**方差/MSE**"。它可解释、不需缩放、能抓非线性，更是**随机森林(4.12)、GBDT(4.13)** 的基石。
> 5.6 covered classification trees. Regression trees are the same idea — split space with yes/no questions — but **each leaf predicts a constant (the mean target in that region)**, and the splitting criterion switches from "impurity" to "**variance/MSE**". Interpretable, scaling-free, captures nonlinearity, and the building block of **Random Forest (4.12), GBDT (4.13)**.
>
> 💼 **实战/面试视角**："回归树怎么切分 / 怎么防过拟合 / 为什么不需缩放" 是表格建模基础。
> 💼 **Practical/interview angle:** "how regression trees split / prevent overfitting / why no scaling" — tabular basics.

> 💡 **面试相关 / Interview-relevant**
> - "回归树的切分准则（方差/MSE 下降）"（出镜率 ★★★★）
> - "回归树 vs 分类树的区别"（★★★★，均值 vs 多数类；MSE vs Gini）
> - "怎么防过拟合（剪枝/深度）"（★★★★★）
> - "为什么树不需缩放"（★★★★）
> - "树为什么不能外推"（★★★，叶子是常数）

---

## 学习目标 / Learning Objectives

1. 理解回归树用**方差/MSE 下降**贪心切分。
   Understand regression trees split greedily by variance/MSE reduction.
2. **从零**实现一棵回归树，对照 sklearn。
   Implement a regression tree from scratch, matching sklearn.
3. 看懂树结构 + 阶梯状（分段常数）预测。
   Read the tree structure and the staircase (piecewise-constant) prediction.
4. 用深度控制过拟合，CV 选深度。
   Control overfitting with depth; choose by CV.
5. 验证树**不需缩放**、读特征重要性。
   Verify trees need no scaling; read feature importances.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [从零回归树 + 阶梯预测 ⭐](#2)
3. [树结构可视化 ⭐](#3)
4. [深度 = 过拟合旋钮 ⭐](#4)
5. [特征重要性 + 不需缩放 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

回归树怎么预测一个连续值？它把特征空间**切成一块块矩形区域**（用一串"某特征 ≤ 阈值"的问题），落在同一区域的所有点**都预测同一个数——该区域内训练样本目标的平均值**。所以它的预测曲线是**阶梯状的分段常数**。
How does a regression tree predict a continuous value? It cuts feature space into **rectangular regions** (via "feature ≤ threshold" questions); all points in a region get **the same prediction — the mean target of the training samples in that region**. So its prediction is a **piecewise-constant staircase**.

怎么决定在哪切？贪心地找让**两边子区域方差(MSE)之和最小**的"特征+阈值"——方差小意味着区域内的目标更一致、预测更准。（对比分类树用 Gini/熵不纯度，5.6。）
How does it choose splits? Greedily find the "feature + threshold" minimizing the **sum of variances (MSE) in the two child regions** — lower variance means more uniform targets, better predictions. (Versus classification trees using Gini/entropy impurity, 5.6.)

继续用 **Diamonds**（4.10 介绍过）。
Continuing with **Diamonds** (introduced in 4.10).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(5000, random_state=0).reset_index(drop=True)
feat_names = ["carat","depth","table","x","y","z"]
X = df[feat_names].values
y = df["price"].values
print(f"Diamonds: {X.shape}, 预测价格")


<a id="2"></a>
## 2. 从零回归树 + 阶梯预测 ⭐ / From Scratch & Staircase

核心是 `best_split`：扫每个候选阈值，算切分后**两边的加权方差**，取最小的那个。叶子的预测值就是落入该叶子样本的**均值**。用单特征画出来，能看到典型的**阶梯状**预测。
The core is `best_split`: scan candidate thresholds, compute the **weighted variance** of the two sides, pick the minimum. A leaf predicts the **mean** of its samples. On one feature, you see the classic **staircase**.


In [ ]:
def best_split(x, y):
    best_mse, best_thresh = np.inf, None
    n = len(y)
    for t in np.unique(x)[1:]:                 # 试每个候选阈值
        L, R = y[x < t], y[x >= t]             # 按阈值切成左右两块
        if len(L) == 0 or len(R) == 0: continue
        mse = (len(L)*L.var() + len(R)*R.var()) / n   # 加权方差(=MSE), 越小越好
        if mse < best_mse:
            best_mse, best_thresh = mse, t
    return best_thresh, best_mse

class SimpleTree:
    def __init__(self, max_depth=3, min_samples=10):
        self.max_depth, self.min_samples = max_depth, min_samples
    def fit(self, x, y, depth=0):
        self.value = y.mean()                   # 叶子预测 = 区域均值
        if depth >= self.max_depth or len(y) < self.min_samples:
            self.leaf = True; return self       # 到达深度/样本下限 → 成为叶子
        t, _ = best_split(x, y)
        if t is None: self.leaf = True; return self
        self.leaf, self.thresh = False, t       # 否则记录切分阈值, 递归建左右子树
        self.left  = SimpleTree(self.max_depth, self.min_samples).fit(x[x<t],  y[x<t],  depth+1)
        self.right = SimpleTree(self.max_depth, self.min_samples).fit(x[x>=t], y[x>=t], depth+1)
        return self
    def predict_one(self, xi):
        if self.leaf: return self.value         # 走到叶子就返回该叶子的均值
        return (self.left if xi < self.thresh else self.right).predict_one(xi)
    def predict(self, x): return np.array([self.predict_one(xi) for xi in x])

xc = df["carat"].values; yc = y
tree = SimpleTree(max_depth=3).fit(xc, yc)
x_plot = np.linspace(xc.min(), xc.max(), 300)
sk = DecisionTreeRegressor(max_depth=3).fit(xc.reshape(-1,1), yc)
print(f"从零树第一刀: carat < {tree.thresh:.3f}")
print(f"sklearn 第一刀阈值: {sk.tree_.threshold[0]:.3f} → 一致(都贪心选 MSE 最小切分)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(xc, yc, alpha=0.1, s=6)
ax.plot(x_plot, tree.predict(x_plot), "r-", lw=2, label="从零树 (depth=3)")
ax.set_xlabel("carat"); ax.set_ylabel("price"); ax.legend()
ax.set_title("回归树: 阶梯状(8 个叶子区域, 每段=区域均值)")
plt.tight_layout(); plt.show()
print("阶梯状预测=分段常数; depth=3 → 最多 2³=8 个叶子区域")


<a id="3"></a>
## 3. 树结构可视化 ⭐ / Visualizing the Tree

`plot_tree` 把树画成流程图——决策树最大的卖点是**可解释**。每个节点显示切分条件、该区域的 MSE（`squared_error`）、样本数、以及预测值（区域均值）。
`plot_tree` draws the tree as a flowchart — trees' biggest selling point is **interpretability**. Each node shows the split condition, the region's MSE (`squared_error`), the sample count, and the predicted value (region mean).


In [ ]:
from sklearn.tree import plot_tree
tree2d = DecisionTreeRegressor(max_depth=2).fit(df[["carat","depth"]], y)   # 浅树便于看清
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(tree2d, feature_names=["carat","depth"], filled=True, rounded=True, fontsize=9, precision=0, ax=ax)
ax.set_title("决策树结构(depth=2): 每节点=切分条件 + 区域均值")
plt.tight_layout(); plt.show()
print("读树: 根节点 carat<阈值? → 左右分支 → 叶子 value=该区域平均价")
print("squared_error=该节点 MSE, samples=落入该节点的样本数")


<a id="4"></a>
## 4. 深度 = 过拟合旋钮 ⭐ / Depth = the Overfitting Knob

`max_depth` 控制复杂度（同分类树 5.6）：太浅→欠拟合（阶梯太粗）；不限深度→每个叶子可能只剩极少样本，**完美记住训练集但严重过拟合**（阶梯锯齿、追噪声）。用 CV 选中等深度，或用 `min_samples_leaf` / `ccp_alpha` 剪枝。
`max_depth` controls complexity (as in 5.6): too shallow → underfit (coarse steps); unbounded → leaves with very few samples, **memorizing the training set but overfitting badly** (jagged, chasing noise). Choose depth by CV, or prune with `min_samples_leaf` / `ccp_alpha`.

> **一个易考点**：树**不能外推**——超出训练数据范围时，它只会输出边缘叶子的那个常数（一条水平线），无法像线性模型那样延伸趋势。
> **A testable point:** trees **can't extrapolate** — beyond the training range they just output the edge leaf's constant (a flat line), unlike linear models that extend trends.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, d in zip(axes, [2, 5, None]):
    t = DecisionTreeRegressor(max_depth=d, random_state=0).fit(xc.reshape(-1,1), yc)
    ax.scatter(xc, yc, alpha=0.08, s=6); ax.plot(x_plot, t.predict(x_plot.reshape(-1,1)), "r-", lw=1.5)
    tag = "欠拟合" if d==2 else ("刚好" if d==5 else "过拟合(每点近一叶)")
    ax.set_title(f"max_depth={d}  {tag}")
plt.tight_layout(); plt.show()

depths = [1,2,3,5,8,12,20,None]
cv = [cross_val_score(DecisionTreeRegressor(max_depth=d, random_state=0), X, y, cv=5, scoring="r2").mean() for d in depths]
print("深度 vs CV R²:")
for d, s in zip(depths, cv): print(f"  max_depth={str(d):<6} CV R²={s:.3f}")
print("深度太浅欠拟合, 不限深度过拟合(CV 掉) → 选中等深度")


<a id="5"></a>
## 5. 特征重要性 + 不需缩放 ⭐ / Importances & No Scaling

**特征重要性** = 该特征在所有切分中带来的总方差下降（归一化）。这里 carat（克拉）主导——钻石价格主要由克拉决定，符合常识。
**Feature importance** = total variance reduction a feature brings across all splits (normalized). Here carat dominates — diamond price is driven by carat, as expected.

并验证树的**缩放不变性**：因为切分只比较"特征 ≤ 阈值"，对特征做任何单调变换（含标准化）结果不变。这是 3.4"树不需缩放"铁律的直接验证。
And we verify trees' **scale-invariance**: since splits only compare "feature ≤ threshold", any monotonic transform (incl. standardizing) leaves the result unchanged — a direct confirmation of the "trees need no scaling" rule from 3.4.


In [ ]:
from sklearn.preprocessing import StandardScaler
tree = DecisionTreeRegressor(max_depth=6, random_state=0).fit(X, y)
imp = pd.Series(tree.feature_importances_, index=feat_names).sort_values(ascending=False)
print("特征重要性(方差下降贡献):")
print(imp.round(3).to_string())
print("carat 主导 — 钻石价格主要由克拉决定\n")

# 验证缩放不变性: 标准化前后 CV R² 应完全相同 / scale-invariance check
raw_r2    = cross_val_score(DecisionTreeRegressor(max_depth=6, random_state=0), X, y, cv=5).mean()
scaled_r2 = cross_val_score(DecisionTreeRegressor(max_depth=6, random_state=0), StandardScaler().fit_transform(X), y, cv=5).mean()
print(f"不缩放 CV R² = {raw_r2:.4f}")
print(f"缩放后 CV R² = {scaled_r2:.4f}")
print("完全相同 → 树对单调变换免疫, 不需缩放(3.4 铁律的验证)")


<a id="6"></a>
## 6. 小结 / Summary

```
回归树: 切空间成矩形区域, 每叶预测=区域目标均值; 预测曲线=阶梯(分段常数)
切分准则: 贪心选让两边加权方差(MSE)最小的(特征,阈值) (分类树用 Gini/熵 5.6)
深度=复杂度: 太浅欠拟合, 不限深度过拟合; CV 选 / min_samples_leaf / ccp_alpha 剪枝
不能外推: 超训练范围只输出边缘叶子的常数(水平线)
特征重要性=总方差下降; 不需缩放(只比阈值, 对单调变换免疫)
单树高方差 → 引出随机森林(4.12)/GBDT(4.13)
```

### 💡 面试速查 / Interview cheat-sheet
1. **切分准则=方差/MSE 下降**(回归), Gini/熵(分类)。
   Split by variance/MSE reduction (regression), Gini/entropy (classification).
2. **叶子预测=区域均值**; 预测是阶梯状分段常数。
   Leaf predicts the region mean; piecewise-constant staircase.
3. **深度防过拟合**(剪枝/min_samples_leaf/ccp_alpha)。
   Depth controls overfitting (pruning/min_samples_leaf/ccp_alpha).
4. **不需缩放**(只比阈值, 对单调变换免疫)。
   No scaling needed (compares thresholds, monotonic-invariant).
5. **不能外推**(叶子常数); 单树高方差 → 用森林/boosting。
   Can't extrapolate (constant leaves); single trees are high-variance → use forests/boosting.

### 下一节 / Next
**4.12 随机森林回归**——单棵树高方差, 用 bagging + 特征随机训很多树取平均, 把方差压下去, 几乎开箱即用。
**4.12 Random Forest Regression** — single trees are high-variance; bagging + feature randomness over many trees averages it away, near-zero-tuning.
